# 📅 2026-08-28 개발 노트 : 코드 정리 + 주간 신작 파이프라인 자동화 + 메인 리디자인

## 🎯 오늘의 목표 — "출시 전 다듬기" (Claude Cowork 협업 세션)

- [x] 코드 전반 이모지 주석/로그 정리 (찐 개발자 톤으로)
- [x] 신작 주기 수집 파이프라인 자동화 (weekly_pipeline + Task Scheduler)
- [x] 메인 히어로 리디자인 — 취향 분석을 전면에
- [x] 네비게이션 바 확대 (작아 보인다는 문제)
- [x] raw 포트폴리오 초안 작성 (docs/portfolio_raw.md)

> 배포는 Vercel(프론트) + Railway(백엔드) 커밋 기반 자동 배포라,
> 오늘 변경분은 로컬 `npm run build` + `pytest` 확인 후 커밋 예정.

## 1. 코드 정리 — 이모지 제거 (43개 파일)

**문제:** 주석/로그에 🚀✅❌ 이모지가 700개+. AI 협업으로 코드를 만들다 보니
생긴 흔적인데, 포트폴리오/코드 공개를 생각하면 프로답지 못한 인상.

**방향:** UI에 보이는 이모지는 남기고 개발자용 텍스트만 정리.
- 제거: 주석(#, //), print/logger/console 로그, docstring 마커, CLI 확인 프롬프트, Django admin 라벨
- 유지: vibe_config의 `emoji` 데이터 필드, 프론트 UI 문자열(✦ 로고, Footer ☕ 등),
  **batch_generator의 SYSTEM_PROMPT** (✅❌가 모델 지시의 일부 — 건드리면 출력 분포 변할 위험),
  cost_guard의 Discord 알람 이모지(🚨/⚠️ — 알람 가시성이 목적이라 기능적)

**구현:** 라인 상태머신 스크립트(triple-quote 블록 스킵, 따옴표 밖 주석만 매칭)로
일괄 처리 + 예외 케이스 수동 패치. `py_compile` 전 파일 통과, `tsc --noEmit` 통과.

**배운 것:** "이모지 다 지워"도 컨텍스트가 필요하다. 프롬프트 문자열과
데이터 필드는 semantic하게 이모지가 '기능'인 경우가 있다.

## 2. 주간 신작 파이프라인 — 오케스트레이터로 묶기

**상황:** 부품은 이미 다 있었음 (Project B 세션에서 만든 것들).
steam_crawler(발견) → batch_generator(few-shot 배치) → batch_processor(적재)
→ generate_embeddings(임베딩) → recalc_percentile(백분위 재계산).
없던 건 "주기적으로 알아서 도는" 글루 코드.

**구현:**
- `embeddings/weekly_pipeline.py` — 5단계를 subprocess로 체이닝.
  단계 실패 시 즉시 중단 + Discord 알림. 신작 0개면 조용히 종료.
  few-shot 파일 없으면 시작 거부 (신작 품질 = few-shot 품질이라 전제조건).
- `scripts/pipeline/setup_weekly_task.ps1` — Windows Task Scheduler 등록
  (매주 월 03:30, 실행 제한 26h — OpenAI Batch 24h 윈도우 감안).
  기존 backup 스케줄러 패턴 재사용.

**막힌 것 + 해결:**
- `batch_generator --upload`가 플래그를 줘도 `input()` 확인을 물어봐서
  무인 실행이 불가능했음 → `--yes` 플래그 추가 (batch_processor에는 이미 있었음).
- 배치 결과 파일 경로를 subprocess 밖에서 알 수 없음
  → 실행 전후 `data/batch_output_*.jsonl` 최신 파일 비교로 해결.

**다음 검증 (커밋 전 필수):**
1. `python -m embeddings.weekly_pipeline --crawl-only` (신작 발견만)
2. fewshot_examples.jsonl 존재 확인 (없으면 fewshot_sampler부터)
3. `--limit 3`으로 소량 end-to-end 1회

## 3. 메인 리디자인 — 취향 분석을 히어로로

**문제의식:** 기존 메인은 로고 + 검색창. "또 하나의 검색 사이트"로 읽힘.
차별점은 취향 분석인데 네비 구석 탭에만 있었음. 네비도 h-14/13px로 작음.

**변경:**
- 히어로: "당신이 좋아할 게임, 데이터는 알고 있어요" 헤드라인
  + "4,190개 × 60개 지표" 신뢰 카피 + **[내 취향 분석하기]** 단일 Primary CTA(/search)
  + 검색창은 그 아래 유지 ("문장으로 검색해보세요" 힌트 추가)
- 네비: h-14→h-16, 탭 13px→14px(px-4 py-2), 로고 확대, 아이콘 버튼 32→36px
- 측정: umami 커스텀 이벤트 `hero_taste_cta`, `nav_taste_click` 추가
  → 배포 후 1주 클릭률 보고 히어로 카피/CTA 위치 조정 (감이 아니라 숫자로)

**남긴 것 (백로그):** 샘플 취향 DNA 카드 렌더(결과 미리보기), 모바일 하단 탭바,
검색 0건 시 vibe 폴백 추천. docs/design_improvements.md 참고.

## 4. 포트폴리오 raw 작성

`docs/portfolio_raw.md` — 회사별로 뽑아쓰는 원재료 방식.
직군별(백엔드/AI·데이터/프론트/스타트업) 조합 가이드 + 수치 테이블 + 트러블슈팅 후보.

**포트폴리오 관점에서 오늘 재발견한 스토리들 (면접용):**
- Few-shot 지식 증류: 교사 GPT-5.4 4,190개 → 학생 gpt-4o-mini, 층화 추출로
  명작 편향 제거(fewshot_sampler), 블라인드 4컬럼 입력으로 편견 차단 → 비용 1/60
- "52 vs 60" 규명: 교사 데이터가 2-패스(31+18)로 만들어진 걸 파이프라인 진단으로 밝혀냄
- custom_id 규약 불일치로 적재 0건 → prefix 무관 숫자 추출로 통일
- stale models.py 지뢰: makemigrations 하면 embedding 컬럼 DROP 위험 → 스키마 정본을
  SQLAlchemy(game.py)로 못박음
- React hydration #418 두 번의 수정, SameSite=None OAuth 쿠키 (git log 참조)

## 📋 다음 할 일

**커밋 전 (이번 변경분):**
- ⬜ 로컬 `npm run build` (Windows에서 — VM엔 SWC 바이너리가 없어 tsc까지만 검증됨)
- ⬜ `pytest` 24개 통과 확인
- ⬜ weekly_pipeline `--crawl-only` → `--limit 3` 소량 리허설
- ⬜ 커밋 분리: ① refactor(emoji 정리) ② feat(weekly pipeline) ③ design(hero/nav)

**그다음:**
- ⬜ setup_weekly_task.ps1 등록 (관리자 PowerShell)
- ⬜ 배포 후 umami로 hero_taste_cta 클릭률 측정 시작
- ⬜ 포트폴리오: 수치 [채우기] 칸 실측으로 채우기 (p95, 캐시 적중률)

**메모:** 4월 이후 신작 백필은 weekly_pipeline을 `--days 150 --limit 100`으로
몇 주에 나눠 돌리면 됨 (비용 상한 감안, 주당 $1 내외).

## 🌙 2차 세션 (같은 날 저녁) — 완성도 올리기

**피드백 반영:** 히어로 카피가 식상하다 → "60개의 세분화된 지표로, 숨은 명작을 찾아냅니다"로 교체, 부제 완전 삭제.

**Steam 로그인 (Phase 2-B 착수):**
- allauth `providers.steam` (OpenID). 핵심 트러블 예상 지점을 선처리:
  Steam은 이메일을 안 주는데 `ACCOUNT_SIGNUP_FIELDS=['email*']`라 자동 가입이 막힘
  → adapter `populate_user`에서 `steam_<id>@users.hiddengem.local` 합성 이메일 주입.
- 닉네임: personaname. setup_oauth.py에 Steam SocialApp 자동 등록 추가 (STEAM_API_KEY).
- requirements에 python3-openid → **django 컨테이너 재빌드 필요.**

**취향 DNA 카드 (Phase 2-C 선행):**
- 신규 의존성 없이 Canvas 2D로 1080×1080 카드 렌더 (조정 지표 바 차트 + 추천작 TOP3).
- Steam CDN 이미지는 CORS taint로 toDataURL이 막혀서 텍스트/도형만으로 디자인.
- 저장 + Web Share API(파일 공유), umami 이벤트 3종. /search 결과 상단에 버튼.

**파이프라인 실가동 점검:** data/fewshot/fewshot_examples.jsonl 이미 존재 (24개, gem 15~95 층화).
남은 건 리허설 2줄: `weekly_pipeline --crawl-only` → `--limit 3`, 그리고 ps1 스케줄 등록.

**다음:** 스와이프 온보딩, Steam 라이브러리 분석(로그인 붙었으니 GetOwnedGames로 확장),
배포 후 dc/루리웹 첫 게시물.